# Projeto Aplicado — Sprint 1 · Base Olist
**Segmentação de Clientes por Valor (RFM e Clusterização) para Otimização de Marketing e Retenção em E-commerce de Moda**

Aluno: Fábio Roberto Gomes Filho · Orientador: Marcos Prochnow · Pós-graduação em Ciência de Dados com IA (XP Educação)

| Sprint 1 | 02/11 – 08/11/2026 |
|---|---|
| R01 | Configurar ambiente Python, repositório GitHub e quadro Trello |
| R02 | Coletar e integrar as tabelas Olist |
| R03 | Limpar e tratar a base Olist |
| R04 | Análise exploratória (EDA) |

> A preparação da base da LLVE (R05) fica em um notebook separado (`PA_Sprint1_LLVE`), para que nenhum dado pessoal apareça neste notebook (LGPD).

---
## R01 · Configuração do ambiente
**Critério de aceite:** ambiente funcional (Drive montado, bibliotecas carregadas, arquivos da Olist localizados) e backlog publicado no Trello.

### 1.1 Montagem do Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Parâmetros do projeto
Todos os caminhos e parâmetros ficam centralizados nesta célula (requisito **RT3 – Reaplicabilidade**).

In [ ]:
from pathlib import Path

# Caminhos no Google Drive
BASE_DIR    = Path('/content/drive/MyDrive/PROJETO APLICADO CD/PROJETO APLICADO')
DATASET_DIR = BASE_DIR / 'DATASET'        # CSVs originais da Olist (somente leitura)
OUTPUT_DIR  = BASE_DIR / 'PROCESSADOS'    # bases tratadas geradas pelos notebooks
FIG_DIR     = BASE_DIR / 'FIGURAS'        # gráficos exportados para o relatório

# Reprodutibilidade (RT1)
SEED = 42

for pasta in (OUTPUT_DIR, FIG_DIR):
    pasta.mkdir(parents=True, exist_ok=True)

print('Base   :', BASE_DIR.exists(), BASE_DIR)
print('Dataset:', DATASET_DIR.exists(), DATASET_DIR)

### 1.3 Bibliotecas e versões

In [ ]:
import sys, platform, random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, scipy

random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')

versoes = pd.DataFrame({
    'componente': ['Python', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'scipy'],
    'versão': [platform.python_version(), pd.__version__, np.__version__, matplotlib.__version__,
               sns.__version__, sklearn.__version__, scipy.__version__],
})
versoes

### 1.4 Inventário dos arquivos da Olist
Confere se os 9 arquivos esperados estão na pasta e registra tamanho, linhas e colunas de cada um.

In [ ]:
ARQUIVOS_OLIST = {
    'orders':        'olist_orders_dataset.csv',
    'order_items':   'olist_order_items_dataset.csv',
    'payments':      'olist_order_payments_dataset.csv',
    'customers':     'olist_customers_dataset.csv',
    'reviews':       'olist_order_reviews_dataset.csv',
    'products':      'olist_products_dataset.csv',
    'sellers':       'olist_sellers_dataset.csv',
    'geolocation':   'olist_geolocation_dataset.csv',
    'translation':   'product_category_name_translation.csv',
}
USADAS_NO_PROJETO = {'orders', 'order_items', 'payments', 'customers'}   # escopo (Quadro 4)

linhas = []
for nome, arq in ARQUIVOS_OLIST.items():
    caminho = DATASET_DIR / arq
    if caminho.exists():
        df_tmp = pd.read_csv(caminho)
        linhas.append({'tabela': nome, 'arquivo': arq,
                       'tamanho_MB': caminho.stat().st_size / 1e6,
                       'linhas': len(df_tmp), 'colunas': df_tmp.shape[1],
                       'usada_no_projeto': nome in USADAS_NO_PROJETO})
    else:
        linhas.append({'tabela': nome, 'arquivo': arq, 'tamanho_MB': None,
                       'linhas': None, 'colunas': None, 'usada_no_projeto': nome in USADAS_NO_PROJETO})

inventario = pd.DataFrame(linhas).astype({'linhas': 'Int64', 'colunas': 'Int64'})
faltando = inventario[inventario['linhas'].isna()]['arquivo'].tolist()
print('Arquivos encontrados:', len(inventario) - len(faltando), 'de', len(inventario))
if faltando:
    print('⚠️ Faltando:', faltando)
inventario

### 1.5 Checklist do R01
- [x] Google Drive montado e pastas `PROCESSADOS` e `FIGURAS` criadas
- [x] Bibliotecas carregadas e versões registradas (RT1)
- [x] Arquivos da Olist localizados e inventariados
- [x] Repositório GitHub criado e este notebook salvo nele (*Arquivo → Salvar uma cópia no GitHub*)
- [x] Quadro Trello com o backlog R01–R16 publicado

---
## R02 · Coleta e integração das tabelas Olist
**Critério de aceite:** base única com **uma linha por pedido entregue**, contendo valor total e `customer_unique_id`.

Fluxo:
1. Carregar as 4 tabelas do escopo (orders, order_items, payments, customers)
2. Validar as chaves e a cardinalidade das relações
3. Agregar itens e pagamentos por pedido
4. Integrar tudo em uma base por pedido e filtrar os pedidos `delivered`
5. Conferir a integridade e salvar em `PROCESSADOS`

### 2.1 Carga das tabelas

In [ ]:
DATAS_ORDERS = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                'order_delivered_customer_date', 'order_estimated_delivery_date']

orders      = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['orders'], parse_dates=DATAS_ORDERS)
order_items = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['order_items'], parse_dates=['shipping_limit_date'])
payments    = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['payments'])
customers   = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['customers'], dtype={'customer_zip_code_prefix': str})

for nome, df in {'orders': orders, 'order_items': order_items, 'payments': payments, 'customers': customers}.items():
    print(f'{nome:<12} {df.shape[0]:>8,} linhas × {df.shape[1]} colunas')

### 2.2 Validação das chaves e da cardinalidade
Ponto crítico do projeto: na Olist, o `customer_id` é gerado **a cada pedido**. O identificador real do cliente é o `customer_unique_id`. Usar o `customer_id` faria todos os clientes parecerem compradores de uma única vez.

In [ ]:
checagens = pd.DataFrame([
    {'verificação': 'order_id único em orders',            'resultado': orders['order_id'].is_unique},
    {'verificação': 'customer_id único em customers',      'resultado': customers['customer_id'].is_unique},
    {'verificação': 'todo pedido tem cliente cadastrado',  'resultado': orders['customer_id'].isin(customers['customer_id']).all()},
    {'verificação': 'relação customer_id → pedido é 1:1',  'resultado': orders['customer_id'].is_unique},
])
display(checagens)

card = pd.DataFrame({
    'medida': ['pedidos (order_id)', 'customer_id distintos', 'clientes reais (customer_unique_id)',
               'itens por pedido (máx.)', 'registros de pagamento por pedido (máx.)'],
    'valor': [orders['order_id'].nunique(), customers['customer_id'].nunique(),
              customers['customer_unique_id'].nunique(),
              order_items.groupby('order_id').size().max(),
              payments.groupby('order_id').size().max()],
})
card

### 2.3 Agregação de itens e pagamentos por pedido
Um pedido pode ter vários itens e vários registros de pagamento (ex.: cartão + voucher). Os dois são consolidados para **uma linha por pedido** antes da integração, evitando duplicar valores no merge.

In [ ]:
itens_pedido = (order_items
    .groupby('order_id')
    .agg(qtd_itens=('order_item_id', 'count'),
         valor_produtos=('price', 'sum'),
         valor_frete=('freight_value', 'sum'))
    .reset_index())

# forma de pagamento principal = a de maior valor no pedido
pag_principal = (payments.sort_values('payment_value', ascending=False)
                 .drop_duplicates('order_id')[['order_id', 'payment_type']]
                 .rename(columns={'payment_type': 'tipo_pagamento_principal'}))

pagamentos_pedido = (payments
    .groupby('order_id')
    .agg(valor_pago=('payment_value', 'sum'),
         qtd_formas_pagamento=('payment_sequential', 'count'),
         parcelas_max=('payment_installments', 'max'))
    .reset_index()
    .merge(pag_principal, on='order_id', how='left'))

print('Pedidos com itens     :', f'{len(itens_pedido):,}')
print('Pedidos com pagamento :', f'{len(pagamentos_pedido):,}')

### 2.4 Integração e filtro de pedidos entregues

In [ ]:
base = (orders
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']],
           on='customer_id', how='left', validate='one_to_one')
    .merge(itens_pedido, on='order_id', how='left', validate='one_to_one')
    .merge(pagamentos_pedido, on='order_id', how='left', validate='one_to_one'))

print('Distribuição de status (todos os pedidos):')
display(base['order_status'].value_counts().to_frame('pedidos')
        .assign(pct=lambda d: d['pedidos'] / d['pedidos'].sum() * 100))

pedidos = base[base['order_status'] == 'delivered'].copy()

funil = pd.DataFrame({
    'etapa': ['Pedidos na base', 'Pedidos entregues (delivered)'],
    'pedidos': [len(base), len(pedidos)],
    'clientes únicos': [base['customer_unique_id'].nunique(), pedidos['customer_unique_id'].nunique()],
})
funil['% dos pedidos'] = funil['pedidos'] / len(base) * 100
funil

### 2.5 Conferência de integridade
- Pedidos entregues sem itens ou sem pagamento
- Diferença entre o valor pago e (produtos + frete). Diferenças pequenas são esperadas por juros de parcelamento e vouchers.

In [ ]:
pedidos['dif_pago_vs_itens'] = pedidos['valor_pago'] - (pedidos['valor_produtos'] + pedidos['valor_frete'])

integridade = pd.DataFrame([
    {'verificação': 'entregues sem itens',                      'pedidos': pedidos['qtd_itens'].isna().sum()},
    {'verificação': 'entregues sem pagamento',                  'pedidos': pedidos['valor_pago'].isna().sum()},
    {'verificação': 'entregues sem data de entrega ao cliente', 'pedidos': pedidos['order_delivered_customer_date'].isna().sum()},
    {'verificação': '|valor pago − (produtos + frete)| > R$ 1', 'pedidos': (pedidos['dif_pago_vs_itens'].abs() > 1).sum()},
])
display(integridade)

print('Resumo financeiro dos pedidos entregues (R$):')
pedidos[['valor_pago', 'valor_produtos', 'valor_frete', 'qtd_itens']].describe().T

### 2.6 Exportação da base integrada

In [ ]:
COLUNAS_BASE = ['order_id', 'customer_unique_id', 'customer_id', 'order_status',
                'order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date',
                'order_estimated_delivery_date', 'customer_city', 'customer_state',
                'qtd_itens', 'valor_produtos', 'valor_frete', 'valor_pago',
                'qtd_formas_pagamento', 'parcelas_max', 'tipo_pagamento_principal']

pedidos = pedidos[COLUNAS_BASE]
arq_saida = OUTPUT_DIR / 'olist_pedidos_integrados.parquet'
pedidos.to_parquet(arq_saida, index=False)

print(f'Salvo: {arq_saida.name}  ·  {pedidos.shape[0]:,} pedidos × {pedidos.shape[1]} colunas  ·  '
      f'{pedidos["customer_unique_id"].nunique():,} clientes únicos')
pedidos.head()

### 2.7 Checklist do R02
- [x] 4 tabelas do escopo carregadas
- [x] Chaves e cardinalidade validadas (`customer_unique_id` como identificador do cliente)
- [x] Itens e pagamentos agregados por pedido, sem duplicação no merge
- [x] Filtro de pedidos `delivered` aplicado e registrado no funil
- [x] Base integrada salva em `PROCESSADOS/olist_pedidos_integrados.parquet`

---
## R03 · Limpeza e tratamento da base Olist
**Critério de aceite:** relatório de qualidade antes e depois do tratamento.

Decisões tomadas (registradas para o relatório):
| # | Decisão | Justificativa |
|---|---|---|
| D1 | Valor monetário (M) = `valor_pago` | Total efetivamente pago (produtos + frete + juros); mesma definição do total de pedido da Nuvemshop (LLVE) |
| D2 | Remover o pedido entregue sem pagamento | Sem valor monetário, não pode compor o RFM |
| D3 | Manter os pedidos sem data de entrega | O RFM usa a data da compra, não a da entrega |
| D4 | Remover clientes com pedido acima do **percentil 99,9** de `valor_pago` | Compras atípicas e raras; o K-Means é sensível a outliers. O cliente sai inteiro para não gerar um falso cliente de baixo valor |

### 3.1 Parâmetros e ponto de partida

In [ ]:
PERCENTIL_CORTE = 0.999   # D4 — corte de valores extremos (RT3: parâmetro configurável)

pedidos = pd.read_parquet(OUTPUT_DIR / 'olist_pedidos_integrados.parquet')

def retrato(df, etapa):
    """Registra o tamanho da base a cada etapa da limpeza."""
    return {'etapa': etapa, 'pedidos': len(df), 'clientes': df['customer_unique_id'].nunique(),
            'receita_R$': df['valor_pago'].sum()}

log_limpeza = [retrato(pedidos, '0. Base integrada (R02)')]
print(f'{len(pedidos):,} pedidos · {pedidos["customer_unique_id"].nunique():,} clientes')

### 3.2 Qualidade antes do tratamento: duplicidades, nulos e domínios

In [ ]:
qualidade = pd.DataFrame({
    'nulos': pedidos.isna().sum(),
    '% nulos': pedidos.isna().mean() * 100,
    'tipo': pedidos.dtypes.astype(str),
})
print('Pedidos duplicados (order_id):', pedidos['order_id'].duplicated().sum())
print('Linhas totalmente duplicadas  :', pedidos.duplicated().sum())
print('Valores pagos ≤ 0             :', (pedidos['valor_pago'] <= 0).sum())
qualidade[qualidade['nulos'] > 0]

### 3.3 Remoção do pedido sem pagamento (D2) e ajuste de tipos

In [ ]:
sem_pagamento = pedidos['valor_pago'].isna()
print('Pedidos sem pagamento removidos:', sem_pagamento.sum())
pedidos = pedidos[~sem_pagamento].copy()
log_limpeza.append(retrato(pedidos, '1. Remoção do pedido sem pagamento'))

# tipos: contagens como inteiros e UF como categoria
for col in ['qtd_itens', 'qtd_formas_pagamento', 'parcelas_max']:
    pedidos[col] = pedidos[col].astype('int64')
pedidos['customer_state'] = pedidos['customer_state'].astype('category')

print('Pedidos sem data de entrega mantidos (D3):', pedidos['order_delivered_customer_date'].isna().sum())

### 3.4 Valores extremos (D4)
O corte é calculado sobre a distribuição de `valor_pago` por pedido. Todo cliente com ao menos um pedido acima do corte é removido por inteiro.

In [ ]:
corte = pedidos['valor_pago'].quantile(PERCENTIL_CORTE)
clientes_extremos = pedidos.loc[pedidos['valor_pago'] > corte, 'customer_unique_id'].unique()
remover = pedidos['customer_unique_id'].isin(clientes_extremos)

impacto = pd.Series({
    'corte (R$)': corte,
    'pedidos acima do corte': (pedidos['valor_pago'] > corte).sum(),
    'clientes removidos': len(clientes_extremos),
    'pedidos removidos (clientes inteiros)': remover.sum(),
    '% dos pedidos': remover.mean() * 100,
    '% da receita': pedidos.loc[remover, 'valor_pago'].sum() / pedidos['valor_pago'].sum() * 100,
}).to_frame('valor')
display(impacto)

antes = pedidos['valor_pago'].copy()
pedidos = pedidos[~remover].copy()
log_limpeza.append(retrato(pedidos, f'2. Remoção de clientes acima do P{PERCENTIL_CORTE*100:g}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, serie, titulo in [(axes[0], antes, 'Antes do tratamento'), (axes[1], pedidos['valor_pago'], 'Depois do tratamento')]:
    sns.boxplot(x=serie, ax=ax, color='#4C9F70', fliersize=2)
    ax.set_xscale('log')
    ax.set_title(f'{titulo} · máx. R$ {serie.max():,.0f}')
    ax.set_xlabel('Valor pago por pedido (R$, escala log)')
axes[0].axvline(corte, color='#C0392B', ls='--', lw=1.2, label=f'corte P99,9 = R$ {corte:,.0f}')
axes[0].legend(loc='lower left', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'R03_outliers_valor_pago.png', dpi=200, bbox_inches='tight')
plt.show()

### 3.5 Qualidade depois do tratamento

In [ ]:
funil_limpeza = pd.DataFrame(log_limpeza)
funil_limpeza['% pedidos retidos'] = funil_limpeza['pedidos'] / funil_limpeza['pedidos'].iloc[0] * 100
funil_limpeza['% receita retida'] = funil_limpeza['receita_R$'] / funil_limpeza['receita_R$'].iloc[0] * 100
display(funil_limpeza)

comparativo = pd.DataFrame({
    'antes': antes.describe(),
    'depois': pedidos['valor_pago'].describe(),
})
comparativo.loc['assimetria'] = [antes.skew(), pedidos['valor_pago'].skew()]
comparativo

### 3.6 Exportação da base tratada

In [ ]:
arq_tratado = OUTPUT_DIR / 'olist_pedidos_tratados.parquet'
pedidos.to_parquet(arq_tratado, index=False)
print(f'Salvo: {arq_tratado.name} · {len(pedidos):,} pedidos · {pedidos["customer_unique_id"].nunique():,} clientes')

### 3.7 Checklist do R03
- [x] Duplicidades, nulos e domínios verificados
- [x] Pedido sem pagamento removido (D2); pedidos sem data de entrega mantidos (D3)
- [x] Tipos ajustados (contagens inteiras, UF categórica)
- [x] Clientes com compras acima do P99,9 removidos por inteiro (D4)
- [x] Funil de limpeza e comparativo antes × depois registrados
- [x] Figura `FIGURAS/R03_outliers_valor_pago.png` exportada
- [x] Base tratada salva em `PROCESSADOS/olist_pedidos_tratados.parquet`